I will reproduce the Gross's 13 and 17 qubit code results

In [4]:
import qutip
import matplotlib
from qutip.piqs.piqs  import *
import numpy as np
import os
os.environ["MOSEKLM_LICENSE_FILE"] = "/Users/46668993/Desktop/qer/mosek/mosek.lic"  # adjust path
from codes.noisemodel import*
from codes.codewords import*
from codes.optimisation import*
import matplotlib.pyplot as plt

In [ ]:
gamma = np.logspace(-5, -2, 10)
dt = 1
p_vals = np.array(gamma) * dt

plt.figure(figsize=(10, 6), dpi = 120)

#bare qubit for comparison
rho_bare = 0.5 * qeye(2)
infids_bare_global = []

#method: kraus or choi
method = 'choi'
method_norecovery = 'kraus'


[l0, l1, _]  = gross_13_piqs()
infids_global = []
infids_local = []
infids_norecovery_global = []
infids_norecovery_local = []

for g in gamma:
    print(f"Processing gamma={g:.2e}")

    try:
        # Global depolarizing
        t0 = time.perf_counter()
        kraus_global = noisemodel('global symmetric depolarizing', N, g, dt, method)
        t1 = time.perf_counter()
        print(f"  Global kraus in {t1 - t0:.2f}s")

        fid_global = optimise(l0, l1, kraus_global, solver='mosek')
        t2 = time.perf_counter()
        print(f"  Global optimise in {t2 - t1:.2f}s")
        
        infids_global.append(abs(1.0 - float(fid_global)))

        #bgm code without recovery operation
        infids_norecovery_global.append(abs(1.0 - float(no_recovery(rho, noisemodel('global symmetric depolarizing',N, g, dt, method_norecovery)))))

        #bare qubit infidelity for comparison
        fid_bare_global = optimise(qutip.basis(2,0), qutip.basis(2,1), noisemodel('global symmetric depolarizing',1,g,dt,method))
        infids_bare_global.append(abs(1.0 - float(fid_bare_global)))
        
        # Delete large objects immediately after use
        del kraus_global
        gc.collect()  # Force garbage collection
        
    except Exception as e:
        print(f"  ERROR (global): {e}")
        infids_global.append(np.nan)

    try:
        # Local depolarizing
        t3 = time.perf_counter()
        kraus_local = noisemodel('local symmetric depolarizing', N, g, dt, method)
        t4 = time.perf_counter()
        print(f"  Local kraus in {t4 - t3:.2f}s")

        fid_local = optimise(l0, l1, kraus_local, solver='mosek')
        t5 = time.perf_counter()
        print(f"  Local optimise in {t5 - t4:.2f}s")
        
        infids_local.append(abs(1.0 - float(fid_local)))

        #bgm code without recovery operation
        infids_norecovery_local.append(abs(1.0 - float(no_recovery(rho, noisemodel('local symmetric depolarizing',N, g, dt, method_norecovery)))))
        
        # Delete large objects immediately after use
        del kraus_local
        gc.collect()  # Force garbage collection
        
    except Exception as e:
        print(f"  ERROR (local): {e}")
        infids_local.append(np.nan)

# Process results
infids_global = np.array(infids_global, dtype=float) #bgm code after recovery
infids_bare_global = np.array(infids_bare_global, dtype=float) #bare qubit
infids_local = np.array(infids_local, dtype=float) #bgm code after recovery
infids_norecovery_global = np.array(infids_norecovery_global, dtype=float)#bgm code without recovery
infids_norecovery_local = np.array(infids_norecovery_local, dtype=float)#bgm code without recovery

mask_global = (infids_global > 0) & np.isfinite(infids_global)
mask_local = (infids_local > 0) & np.isfinite(infids_local)
mask_bare_global = (infids_bare_global > 0) & np.isfinite(infids_bare_global)
mask_infids_norecovery_global = (infids_norecovery_global > 0) & np.isfinite(infids_norecovery_global)
mask_infids_norecovery_local = (infids_norecovery_local > 0) & np.isfinite(infids_norecovery_local)

if np.any(mask_global):
    #plt.loglog(p_vals[mask_global], infids_global[mask_global], "o-", lw=1, label=f"bgm ({b},{g_code},{m}) optimum recovery (global)")
    plt.loglog(p_vals[mask_global], infids_global[mask_global], "o-", lw=1, label=f" optimum recovery (global)")
if np.any(mask_local):
    #plt.loglog(p_vals[mask_local], infids_local[mask_local], "s--", lw=1, label=f"bgm ({b},{g_code},{m}) optimum recovery (local)")
    plt.loglog(p_vals[mask_local], infids_local[mask_local], "s--", lw=1, label=f"optimum recovery (local)")
if np.any(mask_bare_global):
    plt.loglog(p_vals[mask_bare_global], infids_bare_global[mask_bare_global], "o-", lw=1, label="bare qubit")
if np.any(mask_infids_norecovery_global):
    #plt.loglog(p_vals[mask_infids_norecovery_global], infids_norecovery_global[mask_infids_norecovery_global], "x-", lw=1, label=f"bgm ({b},{g_code},{m}) no recovery (global)")
    plt.loglog(p_vals[mask_infids_norecovery_global], infids_norecovery_global[mask_infids_norecovery_global], "x-", lw=1, label=f"no recovery (global)")
if np.any(mask_infids_norecovery_local):
    #plt.loglog(p_vals[mask_infids_norecovery_local], infids_norecovery_local[mask_infids_norecovery_local], "x--", lw=1, label=f"bgm ({b},{g_code},{m}) no recovery (local)")
    plt.loglog(p_vals[mask_infids_norecovery_local], infids_norecovery_local[mask_infids_norecovery_local], "x--", lw=1, label=f"no recovery (local)")

# Clean up large objects for this code
del rho, l0, l1, infids_global, infids_local, infids_bare_global, infids_norecovery_global, infids_norecovery_local
gc.collect()

plt.xlabel("p = " + r"$\gamma \Delta t$")
plt.ylabel("|1 - F|")
plt.title("Performance of (3,3,2) bgm code")
plt.grid(True, which="both", ls="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()